In [ ]:
# ===== 0) Colab 环境设置（本地可跳过）=====
import sys, os, subprocess

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "matplotlib", "pillow", "numpy"], check=False)
    print("Colab 环境已就绪")
else:
    print("本地环境：请确保已安装 torch, transformers, matplotlib, pillow, numpy")

# ffmpeg 检测（Colab 预装）
import shutil
ffmpeg_path = shutil.which("ffmpeg")
print("ffmpeg:", ffmpeg_path if ffmpeg_path else "未找到 — 请安装 ffmpeg")


# 第2讲：多模态大模型与跨模态表征

> 第1讲我们对同一条短视频完成了“转写→摘要”的文本侧处理；  
> 第2讲继续沿用 **同一条视频**，把“视频帧”纳入，做最小闭环任务：  
> **视频帧 × 古籍文本（guji.txt） → 跨模态匹配（相似度矩阵 + Top匹配）**（用于检索/标注/对齐）。

# 本讲结构

## 1  多模态大模型（MLLM）在数字人文的核心应用（仅聚焦：视频-文本匹配）
## 2  跨模态信息融合基础逻辑（简化特征向量概念，侧重应用逻辑）
## 3  LLaVA轻量版适配要点（不展开多模型对比）
## 4  实操：短视频帧 + guji.txt（古籍原文） → 相似度矩阵 → 可视化 → Top 匹配

# 第1章  多模态大模型（MLLM）在数字人文的核心应用
- 本章任务：**视频帧 ↔ 文本片段 的匹配/对齐**
- 本讲的“可交付产物”：相似度矩阵 + Top-K 匹配（便于写作与复现）

# 第1章 MLLM核心应用

## 1.0 从“文本摘要”到“视频-文本匹配”
第1讲解决：转写文本太长、噪声多 → 用摘要做**自动初筛**。  
第2讲解决：即使有了文本，我们仍需要把**画面证据**（视频帧）与**文本证据**（古籍原文片段）对齐。
本讲的数据就是第1讲同一条视频 +  `guji.txt`：
- 视频画面：展示画面
- 文本侧：古籍中关于“饼法/糖蜜/蒸饼/饮茶/煎茶卖之”的原文片段

我们希望得到：哪些画面最像哪些古籍描述（用于检索、标注、对照写作）。

# 第1章 MLLM核心应用

## 1.1 本讲只聚焦：视频-文本匹配
**输入：**
- 视频：抽取若干关键帧（frames）
- 文本：从 `guji.txt` 切出的若干文本片段（texts）

**输出：**
- 相似度分数（cosine similarity）或 Top-K 列表  
  例：Frame_03 最像 Text_12（可能包含“莲花/糖蜜/饼法/饮茶”等关

> 你可以把它当成：跨模态版本的“检索/对齐” baseline。

# 第2章  跨模态信息融合基础逻辑
- 图像帧 → 编码成向量（embedding）
- 文本片段 → 编码成向量（embedding）
- 同一向量空间：语义相近 → 向量更接近
- 最小可用输出：**相似度矩阵（frame × text）**

# 第2章 跨模态信息融合基础逻辑

## 2.0 “特征向量”可以只当成一个可计算的指纹
不用记公式，把它理解成：
- 图像帧 → 一串数字（向量）= 这个画面的“指纹”
- 文本片段 → 一串数字（向量）= 这段文字的“指纹”

同空间的好处：  
我们可以直接算 **指纹相似度**，判断“这帧”和“这段文本”是否讲同一件事。

# 第2章 跨模态信息融合基础逻辑

## 2.1 最小闭环：相似度矩阵（frame × text）
1) 抽帧：video → frames（N张）  
2) 准备文本：guji → texts（M段）  
3) 编码：frames/texts → embeddings（N×D、M×D）  
4) 计算：余弦相似度 → sim（N×M）  
5) 可视化：热力图（heatmap）  
6) Top 匹配：每帧最像哪段文本（或每段文本最像哪帧）

# 第3章  LLaVA轻量版适配要点
- LLaVA 的核心是“视觉编码器 + 语言模型”
- 本讲只做**匹配**（检索/对齐 baseline），不做长文本推理
- 轻量化策略：
  - 少抽帧（3–8 张即可）
  - 文本短一些（每段 1–3 句）
  - 输出固定：相似度矩阵 + Top 匹配（可复现）

> 说明：LLaVA 体系里常用的视觉编码器就是 CLIP 一类模型。  

# 第3章  LLaVA轻量版适配要点

## 3.0 为什么本讲用“LLaVA 的编码思路”先做匹配 baseline？
严格来说，LLaVA 是“看图说话”的视觉语言模型。  
但在工程/科研里，跨模态项目常见的第一步是：
- 先用编码器做 **检索/对齐**（快、稳、可复现）
- 再用生成模型做 **解释/长描述**（更强但更慢）

因此：本讲用 Hugging Face 上通用的“图文编码模型”完成最小任务闭环。  
> 只要你的数据准备好，这个 baseline 就能稳定复现并产出可写进论文/汇报的图表。


# 第4章  实操：短视频帧 + 古籍文本 → 基础跨模态匹配
1) 视频：`西安青年巧塑“盛唐”风味_让千年茶点飘新香.webm.144p.mjpeg.mov`
2) 文本：`guji.txt`（你粘贴整理好的古籍原文拼接文本）

> 要求：把这两个文件放在 notebook 同目录（与 L02.ipynb 同一文件夹）。

In [ ]:
# ===== 4.0) 文件上传（Colab 用户：运行这格上传视频和 guji.txt）=====
import os
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def normalize_guji_filename():
    """Windows 常见：保存为 guji.txt 实际变成 guji.txt.txt。"""
    if Path("guji.txt").exists():
        return
    double_ext = Path("guji.txt.txt")
    if double_ext.exists():
        double_ext.rename("guji.txt")
        print("已自动重命名：guji.txt.txt → guji.txt")
        return
    matches = sorted(Path(".").glob("guji*.txt"))
    if len(matches) == 1:
        matches[0].rename("guji.txt")
        print(f"已自动重命名：{matches[0].name} → guji.txt")

if IN_COLAB:
    print("请上传 2 个文件：")
    print("  1) 视频文件 (.mov / .mp4)")
    print("  2) guji.txt (古籍文本)")
    print("  提示：Windows 用户保存文本时，文件名只填 guji（不要手动加 .txt），避免变成 guji.txt.txt")
    uploaded = files.upload()
    print("已上传:", list(uploaded.keys()))
    normalize_guji_filename()
    if "guji.txt" not in os.listdir("."):
        txt_files = sorted(p.name for p in Path(".").glob("*.txt"))
        print("未检测到 guji.txt，后续会报错。请重新运行这格上传。")
        if txt_files:
            print("当前目录 .txt 文件:", txt_files)
else:
    normalize_guji_filename()
    print("非 Colab 环境：请手动把视频和 guji.txt 放在 L02.ipynb 同目录。")


In [ ]:
# ===== 1) 通用引导：环境自检 + 目录管理 + 设备选择（L01复用）=====
import os, shutil, platform
from pathlib import Path

import torch
import transformers

LESSON_ID = "L02"
OUT_DIR = Path("outputs") / LESSON_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("OS:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

# 设备选择
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("DEVICE =", DEVICE)

# Hugging Face token（如需）
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
print("HF_TOKEN:", "detected" if HF_TOKEN else "(not set)")

# ffmpeg 检测（抽帧需要）
ffmpeg_path = shutil.which("ffmpeg")
print("ffmpeg:", ffmpeg_path)
if ffmpeg_path is None:
    raise RuntimeError(
        "未检测到 ffmpeg。请安装：\n"
        "  本地: conda install -c conda-forge ffmpeg -y\n"
        "  Colab: !apt-get install -y ffmpeg"
    )


In [ ]:
# ===== 2) 选择视频文件 =====
from pathlib import Path

# 如需指定视频，将下行改为：VIDEO_PATH = "/content/your_video.mp4"
VIDEO_PATH = None

cwd = Path(".").resolve()
patterns = ("*.mp4", "*.mov", "*.mkv", "*.webm")

video_files = []
for pat in patterns:
    video_files.extend(cwd.glob(pat))
video_files = sorted(set(video_files))

print("cwd:", cwd)
print("video files found:", [p.name for p in video_files])

if VIDEO_PATH is None:
    if not video_files:
        raise FileNotFoundError(
            "当前目录没有找到视频文件（.mp4 / .mov / .mkv / .webm）。\n"
            "请先运行上方的上传单元格，或将视频文件放到 notebook 同目录。"
        )
    if len(video_files) > 1:
        print("找到多个视频文件，自动选择第一个。如需指定，请将 VIDEO_PATH 设为完整路径。")
    VIDEO_PATH = video_files[0]
else:
    VIDEO_PATH = Path(VIDEO_PATH)
    if not VIDEO_PATH.exists():
        raise FileNotFoundError(f"指定的视频文件不存在：{VIDEO_PATH}")

print("VIDEO_PATH =", VIDEO_PATH)

In [ ]:
# ===== 3) 抽取关键帧（视频 → frames）=====
# 解释：课堂最小可用：每隔几秒抽一张帧，总共抽 N 张（避免复杂镜头检测）

import subprocess
from pathlib import Path

FRAMES_DIR = OUT_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# 清空旧帧
for p in FRAMES_DIR.glob("frame_*.jpg"):
    p.unlink()

FRAME_EVERY_SEC = 2   # 这条短视频节奏快一点，2秒一帧更合适
NUM_FRAMES = 8        # 最多抽 8 帧
SCALE_W = 320         # 你的源视频很低清，320 足够（CLIP会再缩放到模型输入）

out_pattern = str(FRAMES_DIR / "frame_%02d.jpg")

cmd = [
    "ffmpeg", "-y",
    "-i", str(VIDEO_PATH),
    "-vf", f"fps=1/{FRAME_EVERY_SEC},scale={SCALE_W}:-1",
    "-frames:v", str(NUM_FRAMES),
    out_pattern
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

frame_paths = sorted(FRAMES_DIR.glob("frame_*.jpg"))
if not frame_paths:
    raise RuntimeError("抽帧失败：未生成任何 frame_*.jpg，请检查视频是否可读/ffmpeg 是否正常。")

print("frames:", [p.name for p in frame_paths])

如果下一格报错：`ModuleNotFoundError: No module named 'matplotlib'`

请运行下面这格（不要在终端跑）：
```python
!pip install matplotlib -q
```
或者 Colab 用户执行：`!pip install matplotlib -q`

In [ ]:
# ===== 4) 预览关键帧（并保存 montage 方便写作/汇报）=====
import math
import matplotlib.pyplot as plt
from PIL import Image

montage_path = OUT_DIR / "frames_montage.png"

n = len(frame_paths)
cols = min(4, n)
rows = math.ceil(n / cols)

plt.figure(figsize=(3*cols, 3*rows))
for idx, fp in enumerate(frame_paths):
    r = idx + 1
    plt.subplot(rows, cols, r)
    plt.imshow(Image.open(fp))
    plt.axis("off")
    plt.title(fp.stem)

plt.tight_layout()
plt.savefig(montage_path, dpi=160)
plt.show()

print("montage saved:", montage_path)

In [ ]:
# ===== 5) 读取古籍文本（使用你的 guji.txt，并切成候选段落列表）=====
from pathlib import Path
import re

# 如需指定文本文件，将下行改为：TEXT_PATH = "/content/your_text.txt"
TEXT_PATH = None

def resolve_text_path():
    if TEXT_PATH is not None:
        path = Path(TEXT_PATH)
        if not path.exists():
            raise FileNotFoundError(f"指定的文本文件不存在：{path}")
        return path

    for candidate in (Path("guji.txt"), Path("guji.txt.txt")):
        if candidate.exists():
            if candidate.name == "guji.txt.txt":
                print("检测到 guji.txt.txt，将按此文件读取（Windows 双扩展名）")
            return candidate

    txt_files = sorted(Path(".").glob("*.txt"))
    if len(txt_files) == 1:
        print(f"未找到 guji.txt，使用目录中唯一的文本文件：{txt_files[0].name}")
        return txt_files[0]

    if txt_files:
        names = "\n".join(f"  - {p.name}" for p in txt_files)
        raise FileNotFoundError(
            "没找到 guji.txt。请把 guji.txt 放在 notebook 同目录，\n"
            "或将 TEXT_PATH 设为完整路径。当前目录 .txt 文件有：\n" + names
        )
    raise FileNotFoundError("没找到 guji.txt。请把 guji.txt 放在 notebook 同目录。")

TEXT_PATH = resolve_text_path()

raw = TEXT_PATH.read_text(encoding="utf-8", errors="ignore").strip()
print("TEXT_PATH =", TEXT_PATH, "| chars =", len(raw))

# 只做“切段”，不添加任何内容：按中文标点/换行分割
parts = re.split(r"[。！？；\n]+", raw)
parts = [p.strip() for p in parts if p.strip()]

# 过滤太短（避免只有几字像绝句/标题）
parts = [p for p in parts if len(p) >= 18]

# 控制数量（避免热力图太宽）
MAX_TEXTS = 30
texts = parts[:MAX_TEXTS]

MIN_TEXTS = 6  # 建议最少片段数；不足时继续运行，但匹配质量可能下降

if len(texts) == 0:
    raise ValueError(
        "切段后没有可用文本片段。请检查 guji.txt 是否为空，"
        "或段落是否过短（当前过滤阈值：每段至少 18 字）。"
    )

if len(texts) < MIN_TEXTS:
    print(
        f"\n警告：切段后仅有 {len(texts)} 个文本片段（建议至少 {MIN_TEXTS} 个）。\n"
        "  可以继续运行，但相似度矩阵会较窄，Top 匹配结果参考价值可能较低。\n"
        "  建议向 guji.txt 补充更多古籍原文（用 。 或换行分隔），然后重新运行本单元格。\n"
    )

print(f"texts prepared: {len(texts)} segments")
print("Preview (前10条)：")
for i, t in enumerate(texts[:10], start=1):
    print(f"[T{i}] {t}")

In [ ]:
# ===== 6) 加载图文编码模型（轻量 baseline，用于跨模态匹配）=====
# 说明：本讲做的是"匹配/对齐 baseline"（编码→相似度），不做生成式对话。
# 如果模型无法下载/加载，直接报错，不用任何随机向量替代（避免"捏造"）。

import os, torch
from transformers import CLIPProcessor, CLIPModel

CLIP_ID = "openai/clip-vit-base-patch32"

# 国内用户可设置 HF_ENDPOINT 走镜像，如: HF_ENDPOINT=https://hf-mirror.com
# Colab / 海外环境无需设置

try:
    clip_model = CLIPModel.from_pretrained(CLIP_ID)
    clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
except Exception as e:
    raise RuntimeError(
        "CLIP 模型下载/加载失败。请确认：\n"
        "1) 当前环境能访问 Hugging Face，或模型已在本机缓存\n"
        "2) 国内用户可设置环境变量: HF_ENDPOINT=https://hf-mirror.com\n"
        "3) transformers/torch 版本正常\n"
        f"\n原始错误：{e}"
    )

clip_model = clip_model.to(DEVICE)
clip_model.eval()
print("CLIP loaded:", CLIP_ID)


In [ ]:
# ===== 7) 编码（frames/texts → embeddings）并计算相似度矩阵 =====
import numpy as np
import torch
from PIL import Image

images = [Image.open(p).convert("RGB") for p in frame_paths]

with torch.no_grad():
    # 文本向量
    txt_inputs = clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True)
    txt_inputs = {k: v.to(DEVICE) for k, v in txt_inputs.items()}
    txt_out = clip_model.get_text_features(**txt_inputs)
    # transformers 5.x 返回 BaseModelOutputWithPooling，4.x 返回 tensor
    txt_feat = txt_out.pooler_output if hasattr(txt_out, "pooler_output") else txt_out

    # 图像向量
    img_inputs = clip_processor(images=images, return_tensors="pt")
    img_inputs = {k: v.to(DEVICE) for k, v in img_inputs.items()}
    img_out = clip_model.get_image_features(**img_inputs)
    img_feat = img_out.pooler_output if hasattr(img_out, "pooler_output") else img_out

img_emb = img_feat.detach().cpu().numpy()
txt_emb = txt_feat.detach().cpu().numpy()

# 归一化（余弦相似度）
img_emb = img_emb / (np.linalg.norm(img_emb, axis=1, keepdims=True) + 1e-9)
txt_emb = txt_emb / (np.linalg.norm(txt_emb, axis=1, keepdims=True) + 1e-9)

sim = img_emb @ txt_emb.T  # (N_frames, M_texts)

print("sim shape:", sim.shape)
print("preview first frame scores:", np.round(sim[0][:10], 3))


In [ ]:
# ===== 8) 可视化：相似度矩阵热力图（“特征关联性”的最小可视化）=====
import matplotlib.pyplot as plt
import numpy as np

heatmap_path = OUT_DIR / "similarity_heatmap.png"
csv_path = OUT_DIR / "similarity_matrix.csv"

# 保存矩阵（便于复现/写作）
np.savetxt(csv_path, sim, delimiter=",", fmt="%.6f")

plt.figure(figsize=(min(12, 0.35*len(texts)+4), min(10, 0.5*len(frame_paths)+3)))
plt.imshow(sim, aspect="auto")
plt.colorbar(label="cosine similarity")

plt.xticks(range(len(texts)), [f"T{i+1}" for i in range(len(texts))], rotation=90)
plt.yticks(range(len(frame_paths)), [p.stem for p in frame_paths])
plt.title("Frame × Text Similarity (baseline)")

plt.tight_layout()
plt.savefig(heatmap_path, dpi=180)
plt.show()

print("heatmap saved:", heatmap_path)
print("matrix saved:", csv_path)

In [ ]:
# ===== 9) Top 匹配输出：每帧最像哪段文本（并保存为 CSV）=====
import csv
import numpy as np

top_path = OUT_DIR / "top_matches_by_frame.csv"

rows = []
for i, fp in enumerate(frame_paths):
    j = int(np.argmax(sim[i]))
    rows.append({
        "frame": fp.name,
        "best_text_id": f"T{j+1}",
        "score": float(sim[i, j]),
        "text": texts[j]
    })

# 打印前若干条（你也可以全部打印）
print("Top matches (by frame):")
for r in rows[:10]:
    print(f"{r['frame']}  ->  {r['best_text_id']}  score={r['score']:.3f}")

with open(top_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["frame","best_text_id","score","text"])
    w.writeheader()
    w.writerows(rows)

print("saved:", top_path)

In [ ]:
# ===== 10) 反向 Top 匹配：每段文本最像哪帧（可选，但对 DH 叙述更直观）=====
import csv
import numpy as np

top_path2 = OUT_DIR / "top_matches_by_text.csv"

rows2 = []
for j in range(sim.shape[1]):
    i = int(np.argmax(sim[:, j]))
    rows2.append({
        "text_id": f"T{j+1}",
        "best_frame": frame_paths[i].name,
        "score": float(sim[i, j]),
        "text": texts[j]
    })

print("Top matches (by text):")
for r in rows2[:10]:
    print(f"{r['text_id']}  ->  {r['best_frame']}  score={r['score']:.3f}")

with open(top_path2, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["text_id","best_frame","score","text"])
    w.writeheader()
    w.writerows(rows2)

print("saved:", top_path2)

# Quick Check（本讲你应该带走什么）
- **理论**：MLLM 在数字人文里最常见的第一步是“检索/对齐”（视频-文本匹配）
- **方法**：帧/文本编码成向量 → 相似度矩阵 → Top-K（可复现 baseline）
- **本讲真实数据**：西安“盛唐茶点”短视频帧 ×  `guji.txt`（饼法/糖蜜/饮茶等古籍原文）
- **产物**：`frames_montage.png`、`similarity_heatmap.png`、`top_matches_by_frame.csv`、`top_matches_by_text.csv`

> 和第1讲的连贯性：  
> L01 把视频转写文本“变短、可读、可筛”；  
> L02 把“画面证据”与“古籍证据”对齐成可检索的结构化结果（矩阵+Top匹配）。